<img src="https://www.funcionpublica.gov.co/documents/d/guest/logo-universidad-nacional" alt="Logo UNAL" width="600"/>

### **Universidad Nacional de Colombia sede Manizales**
#### Facultad de ingeniería y arquitectura
#### Departamento de ingeniería eléctrica, electrónica y computación
#### *Procesamiento Digital de Imágenes*

#### Profesor: Lucas Iturriago
#### Monitora: Isabella Valero Mora - lvalerom@unal.edu.co

## 1. Configuración Inicial y Preparación de Datos

In [ ]:
!pip install roboflow -q

import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from roboflow import Roboflow
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms.functional as TF
from PIL import Image
import numpy as np

# Configuración del dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

### 1.1. Estructura del Dataset
Definición de las rutas para entrenamiento, validación y prueba.
Se espera que cada partición contenga las imágenes (.jpg) y sus respectivas máscaras (.png).

In [ ]:
rf = Roboflow(api_key="9smAmQgaD8pNOTsDMYKR")
project = rf.workspace("tennis-court-segmentation").project("tennis-court-segmentation-mynwl")
version = project.version(4)
dataset = version.download("png-mask-semantic")

data_dir = "/content/Tennis-Court-Segmentation-4"
train_dir = os.path.join(data_dir, "train")
valid_dir = os.path.join(data_dir, "valid")
test_dir = os.path.join(data_dir, "test")

# Número de clases del problema de segmentación (incluyendo el fondo)
NUM_CLASSES = 2

### 1.2. Pipeline de Datos y Transformaciones
Implementamos un Dataset personalizado que carga tanto la imagen como la máscara.
Se aplican transformaciones espaciales idénticas a ambas para mantener la coherencia.

In [ ]:
class SegmentationDataset(Dataset):
    """
    Dataset de PyTorch para tareas de segmentación semántica.
    """
    def __init__(self, root_dir, is_train=False, target_size=(256, 256)):
        self.root_dir = root_dir
        self.is_train = is_train
        self.target_size = target_size
        
        # Buscamos todas las imágenes .jpg
        self.image_paths = sorted(glob.glob(os.path.join(root_dir, "*.jpg")))
        
    def __len__(self):
        return len(self.image_paths)

    def transform(self, image, mask):
        # 1. Reescalado Obligatorio
        image = TF.resize(image, self.target_size, interpolation=TF.InterpolationMode.BILINEAR)
        # La máscara debe reescalarse con método Nearest Neighbor para no alterar las etiquetas de clase
        mask = TF.resize(mask, self.target_size, interpolation=TF.InterpolationMode.NEAREST)

        # 2. Data Augmentation 'on-the-fly' (solo en entrenamiento)
        if self.is_train:
            # Volteo Horizontal aleatorio
            if np.random.random() > 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)

            # Volteo Vertical aleatorio
            if np.random.random() > 0.5:
                image = TF.vflip(image)
                mask = TF.vflip(mask)
                
            # Rotación aleatoria
            if np.random.random() > 0.5:
                angle = np.random.uniform(-15, 15)
                image = TF.rotate(image, angle, interpolation=TF.InterpolationMode.BILINEAR)
                mask = TF.rotate(mask, angle, interpolation=TF.InterpolationMode.NEAREST)

        # 3. Conversión a Tensores
        image = TF.to_tensor(image)
        # Normalización estándar para imágenes RGB
        image = TF.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        
        # La máscara se convierte a tensor de enteros sin normalizar
        mask = torch.as_tensor(np.array(mask), dtype=torch.long)

        return image, mask

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        # Generar ruta de la máscara correspondiente usando el patrón especificado
        # {nombre_imagen}_mask.png
        base_name = os.path.basename(img_path).replace('.jpg', '')
        mask_path = os.path.join(self.root_dir, f"{base_name}_mask.png")
        
        image = Image.open(img_path).convert("RGB")
        
        if os.path.exists(mask_path):
            mask = Image.open(mask_path)
            # Aseguramos que la máscara sea monocanal para las etiquetas
            if mask.mode != 'L' and mask.mode != 'P':
                mask = mask.convert("L")
        else:
            # Si no existe (ej: inferencia pura sin targets), devolvemos máscara vacía
            mask = Image.new("L", image.size)

        image, mask = self.transform(image, mask)
        return image, mask

# Instanciación de los DataLoaders
try:
    train_dataset = SegmentationDataset(train_dir, is_train=True)
    valid_dataset = SegmentationDataset(valid_dir, is_train=False)
    test_dataset = SegmentationDataset(test_dir, is_train=False)

    batch_size = 8
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    print("Datasets cargados exitosamente.")
except Exception as e:
    print(f"Advertencia al cargar datos: {e}")
    train_loader, valid_loader, test_loader = None, None, None

## 2. Definición de la Arquitectura y Pérdida

### 2.1. Arquitectura U-Net
Implementación de una arquitectura U-Net clásica para segmentación semántica.

In [ ]:
class DoubleConv(nn.Module):
    """Bloque básico de la U-Net: (Conv2d -> BatchNorm -> ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    """Arquitectura U-Net estándar adaptada para segmentación multiclase."""
    def __init__(self, n_channels, n_classes):
        super(UNet, self).__init__()
        
        # Codificador (Ruta de Contracción)
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        
        # Cuello de botella
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))
        
        # Decodificador (Ruta de Expansión)
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(1024, 512)
        
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(512, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 128)
        
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(128, 64)
        
        # Capa de salida
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5)
        # Skip connections
        x = torch.cat([x4, x], dim=1)
        x = self.conv1(x)
        
        x = self.up2(x)
        x = torch.cat([x3, x], dim=1)
        x = self.conv2(x)
        
        x = self.up3(x)
        x = torch.cat([x2, x], dim=1)
        x = self.conv3(x)
        
        x = self.up4(x)
        x = torch.cat([x1, x], dim=1)
        x = self.conv4(x)
        
        logits = self.outc(x)
        return logits

model = UNet(n_channels=3, n_classes=NUM_CLASSES).to(device)

### 2.2. Implementación de Focal Loss para Segmentación
Adaptación de la Focal Loss para tensores de dimensiones espaciales (N, C, H, W).

In [ ]:
class FocalLossSegmentation(nn.Module):
    """
    Focal Loss espacial para problemas de segmentación semántica multiclase.
    """
    def __init__(self, alpha=1.0, gamma=2.0, reduction='mean'):
        super(FocalLossSegmentation, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.ce_loss = nn.CrossEntropyLoss(reduction='none')

    def forward(self, inputs, targets):
        # inputs shape: (N, C, H, W)
        # targets shape: (N, H, W)
        
        ce_loss = self.ce_loss(inputs, targets)
        pt = torch.exp(-ce_loss)
        
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

criterion = FocalLossSegmentation(gamma=2.0)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

## 3. Entrenamiento y Evaluación Continua

### 3.1. Funciones de Métricas (DICE Coefficient)
Cálculo del Coeficiente de Dice por clase y Macro Average para medir el solapamiento.

In [ ]:
def compute_dice_score(y_pred, y_true, num_classes, epsilon=1e-6):
    """
    Calcula el Coeficiente DICE para cada clase sobre un batch.
    """
    # Obtenemos la clase con mayor probabilidad para cada píxel
    preds = torch.argmax(y_pred, dim=1) # Shape: (N, H, W)
    
    dice_per_class = []
    
    for cls in range(num_classes):
        pred_cls = (preds == cls)
        true_cls = (y_true == cls)
        
        intersection = (pred_cls & true_cls).float().sum()
        union = pred_cls.float().sum() + true_cls.float().sum()
        
        dice = (2. * intersection + epsilon) / (union + epsilon)
        dice_per_class.append(dice.item())
        
    macro_dice = np.mean(dice_per_class)
    return macro_dice, dice_per_class

### 3.2. Ciclo de Entrenamiento
Procedimiento para entrenar la U-Net y reportar el desempeño usando la métrica DICE.

In [ ]:
def train_and_validate_seg(model, train_loader, valid_loader, criterion, optimizer, epochs=5):
    """
    Ejecuta el entrenamiento y validación de la red de segmentación.
    """
    if train_loader is None or valid_loader is None:
        print("DataLoaders no configurados. Saltando entrenamiento.")
        return

    for epoch in range(epochs):
        # --- Fase de entrenamiento ---
        model.train()
        train_loss = 0.0
        train_dices = []
        
        for inputs, masks in train_loader:
            inputs, masks = inputs.to(device), masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            macro_dice, _ = compute_dice_score(outputs, masks, NUM_CLASSES)
            train_dices.append(macro_dice)
            
        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_dice = np.mean(train_dices)
        
        # --- Fase de validación ---
        model.eval()
        valid_loss = 0.0
        valid_dices = []
        valid_class_dices = {c: [] for c in range(NUM_CLASSES)}
        
        with torch.no_grad():
            for inputs, masks in valid_loader:
                inputs, masks = inputs.to(device), masks.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, masks)
                
                valid_loss += loss.item()
                macro_dice, class_dices = compute_dice_score(outputs, masks, NUM_CLASSES)
                
                valid_dices.append(macro_dice)
                for c in range(NUM_CLASSES):
                    valid_class_dices[c].append(class_dices[c])
                
        epoch_valid_loss = valid_loss / len(valid_loader)
        epoch_valid_dice = np.mean(valid_dices)
        
        # Reporte por época
        print(f"Época [{epoch+1}/{epochs}]")
        print(f"  Entrenamiento -> Pérdida: {epoch_train_loss:.4f} | Macro DICE: {epoch_train_dice:.4f}")
        print(f"  Validación    -> Pérdida: {epoch_valid_loss:.4f} | Macro DICE: {epoch_valid_dice:.4f}")
        
        # Reporte detallado al final de la última época de validación
        if epoch == epochs - 1:
            print("\n--- Reporte Final de Validación ---")
            print(f"Macro Average DICE: {epoch_valid_dice:.4f}")
            print("DICE por clase:")
            for c in range(NUM_CLASSES):
                class_avg_dice = np.mean(valid_class_dices[c])
                print(f"  Clase {c}: {class_avg_dice:.4f}")

# Para ejecutar el entrenamiento (requiere dataset físico), descomentar:
# train_and_validate_seg(model, train_loader, valid_loader, criterion, optimizer, epochs=5)

## 4. Evaluación Final

### 4.1. Evaluación en el Conjunto de Prueba
Extrae el coeficiente DICE sobre datos no observados para reportar el rendimiento objetivo.

In [ ]:
def evaluate_segmentation_on_test(model, test_loader, num_classes):
    """
    Evalúa la U-Net en el conjunto de prueba, extrayendo métricas DICE detalladas.
    """
    if test_loader is None:
        print("DataLoader de prueba no disponible.")
        return
        
    model.eval()
    test_dices = []
    class_dices_all = {c: [] for c in range(num_classes)}
    
    with torch.no_grad():
        for inputs, masks in test_loader:
            inputs, masks = inputs.to(device), masks.to(device)
            outputs = model(inputs)
            
            macro_dice, class_dices = compute_dice_score(outputs, masks, num_classes)
            test_dices.append(macro_dice)
            
            for c in range(num_classes):
                class_dices_all[c].append(class_dices[c])
                
    final_macro_dice = np.mean(test_dices)
    
    print("\n" + "="*40)
    print("RESULTADOS EN CONJUNTO DE PRUEBA (SEGMENTACIÓN)")
    print("="*40)
    print(f"Macro Average DICE: {final_macro_dice:.4f}\n")
    print("Desglose de Coeficiente DICE por clase:")
    for c in range(num_classes):
        final_class_dice = np.mean(class_dices_all[c])
        print(f"  - Clase {c}: {final_class_dice:.4f}")
    print("="*40)

# Para evaluar en el conjunto de prueba, descomentar:
# evaluate_segmentation_on_test(model, test_loader, NUM_CLASSES)